In [15]:
import torch
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
import torch.nn.functional as F
import torch.nn as nn
import torchvision.transforms as T
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torch.utils.tensorboard import SummaryWriter
import time
import copy
from PIL import Image, ImageFile
from sklearn.metrics import classification_report, confusion_matrix
ImageFile.LOAD_TRUNCATED_IMAGES = True
writer = SummaryWriter()
print('PyTorch version:', torch.__version__)

PyTorch version: 2.11.0


In [16]:
data_dir = './Datasets'
classes  = sorted(os.listdir(data_dir))
print(f'{len(classes)} classes:', classes)

counts = {c: len(os.listdir(os.path.join(data_dir, c))) for c in classes}
for c, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f'  {c:30s}: {n}')
print(f'Total images: {sum(counts.values())}')

9 classes: ['acanthosis-nigricans', 'acne', 'acne-scars', 'alopecia-areata', 'dry', 'melasma', 'oily', 'vitiligo', 'warts']
  dry                           : 804
  acne                          : 363
  warts                         : 306
  melasma                       : 193
  alopecia-areata               : 182
  vitiligo                      : 181
  oily                          : 143
  acanthosis-nigricans          : 50
  acne-scars                    : 49
Total images: 2271


In [17]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224

def get_transforms(mode='train'):
    if mode == 'train':
        return T.Compose([
            T.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
            T.RandomCrop(IMG_SIZE),
            T.RandomHorizontalFlip(0.5),
            T.RandomVerticalFlip(0.2),
            T.RandomRotation(30),
            T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

In [18]:
full_dataset = ImageFolder(data_dir, transform=get_transforms('train'))
print(full_dataset)
print('Class index map:', full_dataset.class_to_idx)

Dataset ImageFolder
    Number of datapoints: 2270
    Root location: ./Datasets
    StandardTransform
Transform: Compose(
               Resize(size=(256, 256), interpolation=bilinear, max_size=None, antialias=True)
               RandomCrop(size=(224, 224), padding=None)
               RandomHorizontalFlip(p=0.5)
               RandomVerticalFlip(p=0.2)
               RandomRotation(degrees=[-30.0, 30.0], interpolation=nearest, expand=False, fill=0)
               ColorJitter(brightness=(0.7, 1.3), contrast=(0.7, 1.3), saturation=(0.8, 1.2), hue=(-0.1, 0.1))
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )
Class index map: {'acanthosis-nigricans': 0, 'acne': 1, 'acne-scars': 2, 'alopecia-areata': 3, 'dry': 4, 'melasma': 5, 'oily': 6, 'vitiligo': 7, 'warts': 8}


In [19]:
torch.manual_seed(42)
total      = len(full_dataset)
val_size   = int(0.10 * total)
test_size  = int(0.10 * total)
train_size = total - val_size - test_size

train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

# Val and test should not use augmentation
val_dataset        = copy.deepcopy(full_dataset)
val_dataset.transform  = get_transforms('val')
test_dataset       = copy.deepcopy(full_dataset)
test_dataset.transform = get_transforms('val')

val_ds  = torch.utils.data.Subset(val_dataset,  val_ds.indices)
test_ds = torch.utils.data.Subset(test_dataset, test_ds.indices)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

Train: 1816  Val: 227  Test: 227


In [20]:
train_labels  = [full_dataset.targets[i] for i in train_ds.indices]
class_counts  = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_wts    = torch.tensor([class_weights[l] for l in train_labels], dtype=torch.float)
sampler       = WeightedRandomSampler(sample_wts, num_samples=len(sample_wts), replacement=True)

print('Class distribution in training set:')
for i, c in enumerate(classes):
    print(f'  {c:30s}: {class_counts[i]:4d}  sample-weight={class_weights[i]:.4f}')

Class distribution in training set:
  acanthosis-nigricans          :   39  sample-weight=0.0256
  acne                          :  296  sample-weight=0.0034
  acne-scars                    :   39  sample-weight=0.0256
  alopecia-areata               :  140  sample-weight=0.0071
  dry                           :  643  sample-weight=0.0016
  melasma                       :  161  sample-weight=0.0062
  oily                          :  112  sample-weight=0.0089
  vitiligo                      :  139  sample-weight=0.0072
  warts                         :  247  sample-weight=0.0040


In [21]:
BATCH_SIZE = 32

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

dataloaders   = {'train': train_dl, 'val': val_dl}
dataset_sizes = {'train': len(train_ds), 'val': len(val_ds)}

In [22]:
efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

efficientnet.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features=1280, out_features=len(classes), bias=True)
)

# Freeze early feature blocks; fine-tune the rest
for param in efficientnet.features[:5].parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in efficientnet.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in efficientnet.parameters())
print(f'Trainable: {trainable:,} / {total_p:,} params')

Trainable: 3,710,417 / 4,019,077 params


In [23]:
# Apple M1/M2 → MPS, NVIDIA → CUDA, fallback → CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
else:
    device = torch.device('cpu')
print('Device:', device)
model = efficientnet.to(device)

Device: mps


In [24]:
NUM_EPOCHS = 30
optimizer  = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

In [25]:
def train_model(model, optimizer, scheduler, num_epochs=30):
    since    = time.time()
    best_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        lr = scheduler.get_last_lr()[0]
        print(f'Epoch {epoch+1}/{num_epochs}  lr={lr:.2e}')
        print('-' * 40)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss     = 0.0
            running_corrects = 0

            for imgs, labels in tqdm(dataloaders[phase], desc=phase, leave=False):
                imgs   = imgs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(imgs)
                    _, preds = torch.max(outputs, 1)
                    loss = F.cross_entropy(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss     += loss.item() * imgs.size(0)
                running_corrects += torch.sum(preds == labels)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc  = running_corrects.float() / dataset_sizes[phase]
            writer.add_scalar(f'Loss/{phase}', epoch_loss, epoch)
            writer.add_scalar(f'Acc/{phase}',  float(epoch_acc), epoch)
            print(f'  {phase:5s}  loss={epoch_loss:.4f}  acc={epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_wts = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), './skin-model-v2-best.pth')
                print(f'  ** Best val acc: {best_acc:.4f} — saved checkpoint')

        scheduler.step()
        print()

    elapsed = time.time() - since
    print(f'Done in {int(elapsed//60)}m {int(elapsed%60)}s  |  Best val acc: {best_acc:.4f}')
    model.load_state_dict(best_wts)
    return model

In [26]:
model = train_model(model, optimizer, scheduler, num_epochs=NUM_EPOCHS)

Epoch 1/30  lr=1.00e-04
----------------------------------------


train:   0%|          | 0/57 [00:00<?, ?it/s]

  train  loss=1.9589  acc=0.3579


  val    loss=1.6812  acc=0.4890
  ** Best val acc: 0.4890 — saved checkpoint

Epoch 2/30  lr=9.97e-05
----------------------------------------


  train  loss=1.3698  acc=0.5837


...

  val    loss=0.7231  acc=0.6960

Epoch 30/30  lr=1.27e-06
----------------------------------------


train:   0%|          | 0/57 [00:00<?, ?it/s]Python(77080) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(77082) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  train  loss=0.3175  acc=0.8623


val:   0%|          | 0/4 [00:00<?, ?it/s]Python(79560) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(79561) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
                                                  

  val    loss=0.6984  acc=0.7004

Done in 68m 7s  |  Best val acc: 0.7137


In [4]:
torch.save(model.state_dict(), './skin-model-v2-best.pth')
print('Model weights saved to skin-model-v2.pth')

NameError: name 'torch' is not defined

In [5]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_dl, desc='Testing'):
        imgs    = imgs.to(device)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print('\n=== Per-class report on held-out test set ===')
print(classification_report(all_labels, all_preds, target_names=classes))

NameError: name 'model' is not defined

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=classes, yticklabels=classes, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.savefig('./confusion_matrix.png', dpi=150)
plt.show()
print('Saved confusion_matrix.png')